In [4]:
import sys

In [5]:
sys.path.append('../')
import utils_frozen as utils

In [6]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [7]:
X = pd.read_csv("data/split_a.csv", header=None, index_col=0)
Y = pd.read_csv("data/split_b.csv", header=None, index_col=0)
X_np = X.to_numpy()
Y_np = Y.to_numpy()
gene_count = X_np.shape[0]
print(gene_count)

3291


In [8]:
# x numpy vector of counts
# transformation to apply to the non-zero elements of x.
def process_counts(x, transformation):
    cell_count = len(x)
    N_0 = np.sum(x == 0)
    N_plus = cell_count - N_0
    t_x = transformation(x[np.where(x > 0)])    
    return(t_x, N_plus, N_0)

In [9]:
# Define transformations here: 
# [1] https://www.10xgenomics.com/analysis-guides/single-cell-rna-seq-data-normalization
# [2] https://www.nature.com/articles/s41592-023-01814-1#Sec6

# Standard transformation but no +1 typically used in other packages e.g., Seurat (https://satijalab.org/seurat/reference/normalizedata).
def log_transform(x, scale_factor=10000):
    norm_x = np.log(scale_factor * x/x.sum(0))
    return(norm_x)
    

In [15]:
ci_x = list(map(lambda x: utils.get_intervals(*process_counts(x, log_transform), z=1.96, model="lognormal"), X_np))

In [16]:
ci_y = list(map(lambda x: utils.get_intervals(*process_counts(x, log_transform), z=1.96, model="lognormal"), Y_np))

In [19]:
# return p-value -- we are using Normal assumption on the means for inference. The null is that the means are the same.
def infer_lfc(m_x, s_x, m_y, s_y):
    diff = m_x - m_y 
    s_hat = np.sqrt((s_x**2) + (s_y**2))
    lb = diff - 1.96*s_hat
    ub = diff + 1.96*s_hat
    # Compute the p-value.
    z_score = diff / s_hat
    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
    return(lb,ub,p_value)


In [20]:
infer_results = [infer_lfc(xx[1], xx[2], yy[1], yy[2]) for xx, yy in zip(ci_x, ci_y)]
infer_results = np.asarray(infer_results)

In [25]:
lb = infer_results[:, 0]
ub = infer_results[:, 1]
pvals = infer_results[:, 2]
# Calculate the count of results where 0 is within the interval
rejection_count = np.sum(pvals[~np.isnan(pvals)] < 0.05)
print(rejection_count)
rejection_prop = rejection_count / gene_count
print(rejection_prop)

10
0.003038590094196293


In [26]:
import statsmodels.stats.multitest as smm

In [27]:
adjusted_p_values = smm.multipletests(pvals[~np.isnan(pvals)], alpha=0.05, method='bonferroni')[1]

print("Bonferroni-adjusted rejection rate:", np.mean(adjusted_p_values < 0.05))



FDR: 0.0006090133982947625
